In [55]:
import pandas as pd
import numpy as np

In [74]:
df = pd.read_csv('/PyAi/dataset_dt_2kelas_3var_100_noisy.csv')
df.head()

,id,luas_tanah,luas_bangunan,jarak_kota,kelas
0,R084,211,62,8.2,Murah
1,R054,106,72,5.1,Murah
2,R071,71,48,8.7,Murah
3,R046,218,165,6.9,Mahal
4,R045,66,35,12.1,Murah


In [75]:
# GANTI sesuai kolom dataset kamu
FEATURES = ["luas_tanah", "luas_bangunan", "jarak_kota"]
TARGET = "kelas"

X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy()

print("Jumlah data:", len(df))
print("Fitur:", FEATURES)
print("Target:", TARGET)
print("Contoh label unik:", np.unique(y))

Jumlah data: 100
Fitur: ['luas_tanah', 'luas_bangunan', 'jarak_kota']
Target: kelas
Contoh label unik: ['Mahal' 'Murah']


In [76]:
def train_test_split(X, y, test_size=0.2, seed=42):
    """Split data acak: train & test."""
    rng = np.random.default_rng(seed)
    idx = np.arange(len(X))
    rng.shuffle(idx)
    split = int(len(X) * (1 - test_size))
    train_idx, test_idx = idx[:split], idx[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def gini(y):
    """Gini impurity."""
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)

def majority_class(y):
    """Kelas mayoritas (untuk leaf)."""
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]

def accuracy(y_true, y_pred):
    """Akurasi sederhana."""
    return np.mean(y_true == y_pred)

In [77]:
class Node:
    """Satu simpul (node) di decision tree.

    Jika node adalah leaf:
      - value = kelas prediksi

    Jika node adalah decision node:
      - feature_idx = indeks fitur untuk split
      - threshold   = nilai batas split
      - left/right  = anak kiri/kanan
    """
    def __init__(self, *, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [78]:
class DecisionTreeClassifierScratch:
    def __init__(self, max_depth=5, min_samples_split=2, min_impurity_decrease=1e-7):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_impurity_decrease = min_impurity_decrease
        self.root = None

    def fit(self, X, y):
        """Bangun tree dari data latih."""
        self.root = self._build_tree(X, y, depth=0)

    def predict(self, X):
        """Prediksi kelas untuk banyak data."""
        return np.array([self._predict_one(x, self.root) for x in X])

    def _predict_one(self, x, node):
        """Prediksi kelas untuk 1 baris data."""
        while node.value is None:  # selama belum leaf
            if x[node.feature_idx] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

In [79]:
def best_split(X, y):
    """Mengembalikan (best_feature, best_threshold, best_gain)."""
    n_samples, n_features = X.shape
    parent_impurity = gini(y)

    best_gain = 0.0
    best_feature = None
    best_thresh = None

    for f in range(n_features):
        values = np.unique(X[:, f])
        if len(values) == 1:
            continue

        # kandidat threshold = midpoint dari nilai unik yang sudah diurutkan
        thresholds = (values[:-1] + values[1:]) / 2.0

        for t in thresholds:
            left_mask = X[:, f] <= t
            right_mask = ~left_mask

            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue

            y_left, y_right = y[left_mask], y[right_mask]

            # impurity setelah split (weighted)
            w_left = len(y_left) / n_samples
            w_right = len(y_right) / n_samples
            child_impurity = w_left * gini(y_left) + w_right * gini(y_right)

            gain = parent_impurity - child_impurity

            if gain > best_gain:
                best_gain = gain
                best_feature = f
                best_thresh = t

    return best_feature, best_thresh, best_gain

In [80]:
def _best_split(self, X, y):
    return best_split(X, y)

DecisionTreeClassifierScratch._best_split = _best_split

In [81]:
def _build_tree(self, X, y, depth):
    # (A) Stop jika semua label sama => leaf
    if len(np.unique(y)) == 1:
        return Node(value=y[0])

    # (B) Stop jika depth mentok / data terlalu sedikit => leaf mayoritas
    if depth >= self.max_depth or len(y) < self.min_samples_split:
        return Node(value=majority_class(y))

    # (C) Cari split terbaik
    feature, thresh, gain = self._best_split(X, y)

    # (D) Jika tidak ada split bagus => leaf mayoritas
    if feature is None or gain < self.min_impurity_decrease:
        return Node(value=majority_class(y))

    # (E) Split data
    left_mask = X[:, feature] <= thresh
    right_mask = ~left_mask

    left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
    right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

    return Node(feature_idx=feature, threshold=thresh, left=left, right=right)

DecisionTreeClassifierScratch._build_tree = _build_tree

In [82]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, seed=123)

# Latih model
tree = DecisionTreeClassifierScratch(max_depth=4, min_samples_split=3)
tree.fit(X_train, y_train)

# Prediksi
y_pred = tree.predict(X_test)

print("Accuracy:", accuracy(y_test, y_pred))

# Lihat 10 hasil pertama
pd.DataFrame({"y_true": y_test[:10], "y_pred": y_pred[:10]})

Accuracy: 0.4666666666666667


,y_true,y_pred
0,Murah,Mahal
1,Mahal,Murah
2,Mahal,Mahal
3,Mahal,Murah
4,Mahal,Murah
5,Mahal,Murah
6,Murah,Murah
7,Murah,Murah
8,Murah,Murah
9,Mahal,Murah


In [83]:
def confusion_matrix_df(y_true, y_pred, labels=None):
    """Confusion matrix dalam bentuk DataFrame."""
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    labels = list(labels)
    label_to_idx = {lab: i for i, lab in enumerate(labels)}
    cm = np.zeros((len(labels), len(labels)), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        cm[label_to_idx[yt], label_to_idx[yp]] += 1
    return pd.DataFrame(cm, index=[f"True_{l}" for l in labels], columns=[f"Pred_{l}" for l in labels])

def classification_report_df(y_true, y_pred, labels=None):
    """Hitung precision/recall/F1 per kelas + macro & weighted avg."""
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    labels = list(labels)
    
    rows = []
    support_total = 0
    macro_p = macro_r = macro_f1 = 0.0
    weighted_p = weighted_r = weighted_f1 = 0.0
    
    for lab in labels:
        tp = np.sum((y_true == lab) & (y_pred == lab))
        fp = np.sum((y_true != lab) & (y_pred == lab))
        fn = np.sum((y_true == lab) & (y_pred != lab))
        support = np.sum(y_true == lab)
        support_total += support
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        
        rows.append([lab, precision, recall, f1, support])
        
        macro_p += precision
        macro_r += recall
        macro_f1 += f1
        
        weighted_p += precision * support
        weighted_r += recall * support
        weighted_f1 += f1 * support
    
    n_classes = len(labels)
    macro_p /= n_classes
    macro_r /= n_classes
    macro_f1 /= n_classes
    
    if support_total > 0:
        weighted_p /= support_total
        weighted_r /= support_total
        weighted_f1 /= support_total
    
    report = pd.DataFrame(rows, columns=["class", "precision", "recall", "f1_score", "support"])
    
    # Tambahkan ringkasan
    summary = pd.DataFrame([
        ["macro_avg", macro_p, macro_r, macro_f1, support_total],
        ["weighted_avg", weighted_p, weighted_r, weighted_f1, support_total],
    ], columns=report.columns)
    
    report = pd.concat([report, summary], ignore_index=True)
    return report

# --- Hitung metrik ---
labels = np.unique(np.concatenate([y_test, y_pred]))
cm = confusion_matrix_df(y_test, y_pred, labels=labels)
report = classification_report_df(y_test, y_pred, labels=labels)

print("Akurasi:", accuracy(y_test, y_pred))
print("\nConfusion Matrix:")
display(cm)

print("\nPrecision / Recall / F1 per kelas:")
display(report)

Akurasi: 0.4666666666666667

Confusion Matrix:


,Pred_Mahal,Pred_Murah
True_Mahal,7,15
True_Murah,1,7



Precision / Recall / F1 per kelas:


,class,precision,recall,f1_score,support
0,Mahal,0.875000,0.318182,0.466667,22
1,Murah,0.318182,0.875000,0.466667,8
2,macro_avg,0.596591,0.596591,0.466667,30
3,weighted_avg,0.726515,0.466667,0.466667,30


In [84]:
# Contoh input baru: [luas_tanah_m2, luas_bangunan_m2]
contoh = np.array([
    [80, 90, 88],
    [150, 110, 100],
], dtype=float)

pred_contoh = tree.predict(contoh)
pd.DataFrame(contoh, columns=FEATURES).assign(prediksi_kelas=pred_contoh)

,luas_tanah,luas_bangunan,jarak_kota,prediksi_kelas
0,80.0,90.0,88.0,Murah
1,150.0,110.0,100.0,Murah
